# Session 9: charts

The data is clean. Now make it say something.

In a notebook, charts appear under the cell that made them, which is why
notebooks are so good for this work. `savefig` still matters when the chart
has to go into a document, an email or a repo.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

here = Path.cwd()
while not (here / "data" / "messy").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

# The cleaning function from the end of the previous notebook. Copied here,
# because notebooks do not share variables with each other.
#
# Copying it is exactly the itch session 10 scratches: put it in a .py file
# once, import it everywhere, fix it in one place.
def clean_plays(messy):
    """Return a cleaned copy of the messy listening log."""
    df = messy.rename(columns={"minutes played ": "minutes_played"}).copy()

    for column in ["artist_name", "track_name", "genre", "country", "device"]:
        df[column] = df[column].str.strip()
    df["genre"] = df["genre"].str.title()
    df["device"] = df["device"].str.lower()

    df["minutes_played"] = pd.to_numeric(
        df["minutes_played"].str.replace(" min", "", regex=False)
                            .str.replace(" ", "", regex=False),
        errors="coerce",
    )

    raw = df["played_at"].str.strip()
    df["played_at"] = (
        pd.to_datetime(raw, format="%Y-%m-%d", errors="coerce")
        .fillna(pd.to_datetime(raw, format="%d/%m/%Y", errors="coerce"))
        .fillna(pd.to_datetime(raw, format="%d-%m-%Y", errors="coerce"))
    )

    df = df.drop_duplicates().dropna(subset=["minutes_played"])
    df["device"] = df["device"].fillna("unknown")
    df["genre"] = df["genre"].fillna("Unknown")
    df["country"] = df["country"].fillna("Unknown")
    return df.reset_index(drop=True)


plays = clean_plays(pd.read_csv("data/messy/plays_messy.csv"))
print(plays.shape)
plays.head(3)

## The shape of every chart you will make

In [ ]:
monthly = plays.set_index("played_at")["minutes_played"].resample("MS").sum()

fig, ax = plt.subplots(figsize=(9, 4.5))

ax.plot(monthly.index, monthly.values, marker="o", linewidth=2)

ax.set_title("Listening drops off every summer")
ax.set_xlabel("Month")
ax.set_ylabel("Minutes played")
ax.set_ylim(bottom=0)
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
plt.show()

**Two objects, and the difference matters.**

* the **figure** is the picture: its size, and saving it
* the **axes** are the plot inside it: the data, the labels, the limits

Almost everything you do is `ax.something()`.

You will see tutorials using `plt.plot()` with no `fig` or `ax`. It works,
and it breaks down the moment you want two charts. Learn the `fig, ax`
version once and stop thinking about it.

`resample("MS")` groups a date index by month start. It only works because
the dates are real dates, which is what the cleaning was for.

## Four charts, and when each is right

| Chart | Use it for |
|---|---|
| line, `ax.plot` | change over time |
| bar, `ax.bar` | comparing categories |
| scatter, `ax.scatter` | relationship between two numbers |
| histogram, `ax.hist` | distribution of one number |

### Bar: comparing categories

Sorted by value, because alphabetical order is almost never the order the
reader wants.

In [ ]:
by_genre = plays.groupby("genre")["minutes_played"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(by_genre.index, by_genre.values, color="#4f6ddb")

ax.set_title("Electronic accounts for almost a third of all listening")
ax.set_xlabel("Genre")
ax.set_ylabel("Total minutes")
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
plt.show()

print((by_genre / by_genre.sum() * 100).round(1).head(3))

### Histogram: the distribution of one number

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(plays["minutes_played"], bins=30, color="#37d3a6", edgecolor="white")

ax.set_title("Most plays last three to six minutes, with a spike of very short ones")
ax.set_xlabel("Minutes played")
ax.set_ylabel("Number of plays")
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
plt.show()

That spike on the left looks like it might be the skipped plays. Worth
checking rather than assuming, which is what the next chart does.

### Two series on one chart, which needs a legend

In [ ]:
kept = plays[plays["skipped"] == 0]["minutes_played"]
skipped = plays[plays["skipped"] == 1]["minutes_played"]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist([kept, skipped], bins=25, stacked=True,
        color=["#4f6ddb", "#ff7085"], label=["played through", "skipped"])

ax.set_title("Nearly every skip is under two minutes")
ax.set_xlabel("Minutes played")
ax.set_ylabel("Number of plays")
ax.legend()
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
plt.show()

print(f"skipped plays average {skipped.mean():.2f} minutes, "
      f"the rest average {kept.mean():.2f}")
print(f"share of skips under two minutes:      {(skipped < 2).mean() * 100:.0f}%")
print(f"share of plays under two minutes that "
      f"were skips: {(plays[plays['minutes_played'] < 2]['skipped']).mean() * 100:.0f}%")

**A legend is not optional once there is more than one series.** Without it
the chart shows two colours and no way to tell which is which.

## And read those two numbers carefully

98% of skips are under two minutes. Only 64% of plays under two minutes are
skips. Those are different statements, and it is very easy to see this chart
and come away with the second one when the data supports the first.

The chart title says the true, narrow thing. "Short plays are skips" would
have been the same chart, the same colours, and a claim the data does not
make.

### Scatter: the relationship between two numbers

In [ ]:
per_artist = (plays.groupby("artist_name")
                   .agg(plays=("play_id", "count"),
                        minutes=("minutes_played", "sum")))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(per_artist["plays"], per_artist["minutes"], s=40, alpha=0.75)

ax.set_title("More plays means more minutes, but the slope varies a lot")
ax.set_xlabel("Number of plays")
ax.set_ylabel("Total minutes")
ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

The spread around that line is the interesting part: two artists with the
same number of plays can differ a lot in total minutes, because their track
lengths differ. That is the total-versus-average lesson again, drawn.

---

# Labels: a chart without them is not finished

An unlabelled chart says: "here is a shape, you work out what it means, what
the units are, and when it is from". Nobody reading it can check whether you
are right, which means nobody should believe it.

**The minimum, every time:**

* a **title** that says the finding, not the columns
* both **axis labels**, with units
* a **legend**, if there is more than one series

## Titles: say the finding

`"Minutes by month"` tells the reader what the axes already tell them.
`"Listening drops off every summer"` tells them what you found.

If you cannot write the second kind, you may not have found anything yet, and
that is useful to notice.

Here is the claim in that first title, checked:

In [ ]:
summer = monthly[monthly.index.month.isin([6, 7, 8])]
rest = monthly[~monthly.index.month.isin([6, 7, 8])]
print(f"summer months average {summer.mean():.0f} minutes")
print(f"other months average  {rest.mean():.0f} minutes")

The title is doing honest work: summer really is about 40% quieter.

---

# The easiest way to mislead with a chart

The same four numbers, twice.

In [ ]:
avg = plays.groupby("device")["minutes_played"].mean().sort_values(ascending=False)

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))

left.bar(avg.index, avg.values, color="#4f6ddb")
left.set_ylim(bottom=0)
left.set_title("Honest: axis starts at zero")
left.set_ylabel("Average minutes")

right.bar(avg.index, avg.values, color="#ff7085")
right.set_ylim(3.9, 4.4)
right.set_title("Same data, cropped axis")
right.set_ylabel("Average minutes")

fig.tight_layout()
plt.show()

print(avg.round(2))

On the left, four bars of nearly identical height, because the difference
between the highest and lowest is about 9%. On the right, the car towers over
the speaker and it looks like a finding.

**Bar charts should start at zero.** The length of a bar *is* the quantity,
so cutting the bottom off lies about the ratio. Line charts have more
latitude, since they are about change rather than magnitude, but say so if
you crop one.

This happens by accident at least as often as on purpose, because some
plotting tools crop by default to "use the space well". Check your y-axis
before you send a chart to anybody.

---

# Saving

In a notebook the chart appears by itself. To keep one, save it.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(monthly.index, monthly.values, marker="o", linewidth=2)
ax.set_title("Listening drops off every summer")
ax.set_xlabel("Month")
ax.set_ylabel("Minutes played")
ax.set_ylim(bottom=0)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()

fig.savefig(OUTPUT / "minutes_by_month.png", dpi=150)
plt.close(fig)          # free the memory, and stop it displaying twice

print("saved to", OUTPUT / "minutes_by_month.png")

`dpi=150` is a good default: sharp on a screen, small enough to email.

**In a script**, `plt.show()` opens a window and *waits for you to close
it*. That is fine once and useless in anything scheduled, where nobody is
there to click. Two lines prevent it:

```python
import matplotlib
matplotlib.use("Agg")      # save files, never open a window
```

`output/` is in `.gitignore`, because a generated file can always be
regenerated. Commit the code that makes the chart, not the chart.

`sessions/session-09/demos/clean_and_chart.py` is this notebook as a script,
and it is worth a look: it is nearly the pipeline you will build next
session.